# Phase 13.8 — Serving Performance

13.8.1–13.8.5 serving benchmarks; 13.8.6 concurrency sweeps; 13.8.7 baseline/regression (opt-in; never fabricate measurements).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from productiq.serving.performance.search_workloads import SEARCH_SERVING_PERFORMANCE_WORKLOADS

[(w.workload_id, w.workload_name) for w in SEARCH_SERVING_PERFORMANCE_WORKLOADS]

## Production search benchmark (opt-in)

Requires PostgreSQL/pgvector, production retrieval smoke flags, and `PRODUCTIQ_SEARCH_SERVING_PERFORMANCE_BENCHMARK=1`.

```bash
python scripts/run_search_serving_performance_benchmark.py
```

Artifacts: `resources/benchmark/search_serving/`. Do not commit environment-specific latency JSON unless intentional.

In [ ]:
from productiq.serving.performance.recommendation_workloads import RECOMMENDATION_SERVING_PERFORMANCE_WORKLOADS

[(w.workload_id, w.request_body.get("seed_product_id")) for w in RECOMMENDATION_SERVING_PERFORMANCE_WORKLOADS]

## Production recommendation benchmark (opt-in)

Requires the same PostgreSQL/BM25 prerequisites as search production benchmarks plus `PRODUCTIQ_RECOMMENDATION_SERVING_PERFORMANCE_BENCHMARK=1`. Measures `POST /api/v1/recommendations` (SIMILAR) and `ProductionRecommendationServingService`.

```bash
python scripts/run_recommendation_serving_performance_benchmark.py
```

Artifacts: `resources/benchmark/recommendation_serving/`. Run locally to capture measurements; never fabricate numbers in docs.

In [ ]:
from tests.recommendation.test_recommendation_pipeline import _build_catalog, _build_pipeline
from productiq.serving.performance.recommendation_benchmark import run_recommendation_serving_performance_suite
from productiq.serving.performance.recommendation_benchmark_config import RecommendationServingBenchmarkSettings
from productiq.serving.performance.recommendation_workloads import RECOMMENDATION_SERVING_BRAND_ACTIVITY_V1

seed = str(RECOMMENDATION_SERVING_BRAND_ACTIVITY_V1.request_body["seed_product_id"])
catalog, filtering = _build_catalog(seed, "460825114005", "460811249001")
pipeline = _build_pipeline(catalog, filtering)
suite = run_recommendation_serving_performance_suite(
    pipeline,
    used_full_bm25_index=False,
    settings=RecommendationServingBenchmarkSettings(warmups=0, iterations=1, concurrency=1),
    workloads=(RECOMMENDATION_SERVING_BRAND_ACTIVITY_V1,),
)
suite.service_results[0].latency.mean_ms if suite.service_results[0].latency else None

In [ ]:
from tests.retrieval.test_production_ranking_integration import _ranked_pipeline
from productiq.serving.performance.search_benchmark import run_search_serving_performance_suite
from productiq.serving.performance.search_benchmark_config import SearchServingBenchmarkSettings
from productiq.serving.performance.search_workloads import SEARCH_SERVING_LEXICAL_V1

pipeline, _, _ = _ranked_pipeline(ranking=("P1", "P2"), pool_top_k=10)
suite = run_search_serving_performance_suite(
    pipeline,
    used_full_bm25_index=False,
    candidate_pool_top_k=10,
    settings=SearchServingBenchmarkSettings(warmups=0, iterations=2, concurrency=1),
    workloads=(SEARCH_SERVING_LEXICAL_V1,),
)
suite.api_results[0].errors.model_dump(), suite.service_results[0].latency.p50_ms if suite.service_results[0].latency else None

## Baseline & regression (13.8.7)

Synthetic comparison only — pin a baseline from one run, compare a second run with identical configuration. No production numbers fabricated.

In [ ]:
from productiq.serving.performance.regression_engine import compare_to_baseline_record
from productiq.serving.performance.baseline_schema import ServingPerformanceBaselineRecord
from productiq.serving.performance.baseline_identity import identity_from_run_result
from datetime import UTC, datetime
from tests.serving.test_serving_performance_baseline_regression import _sample_run, _baseline_record

baseline_run = _sample_run(run_id="nb-baseline", p50=10.0)
current_run = _sample_run(run_id="nb-current", p50=12.0)
report = compare_to_baseline_record(_baseline_record(run=baseline_run), current_run)
{
    "compatible": report.compatibility.compatible,
    "mean_delta_ms": report.latency_comparison.absolute_delta_ms["mean_ms"]
    if report.latency_comparison
    else None,
    "regression_flags": report.regression_flags,
}

## Evidence-based optimization (13.8.8)

Inventory real artifacts, classify findings (OBSERVED / POTENTIAL / UNKNOWN), and only optimize when a benchmark proves a bottleneck. This repository state blocked production changes (missing 13.8.3/13.8.4 runs; product service already sub-ms).

In [ ]:
from productiq.serving.performance.evidence_analysis import (
    build_serving_performance_evidence_report,
    render_evidence_report_markdown,
)

evidence = build_serving_performance_evidence_report()
{
    "optimization_selected": evidence.optimization_selected,
    "optimization_retained": evidence.optimization_retained,
    "blockers": evidence.optimization_blockers,
    "search_artifacts": evidence.inventory.search_serving_run_json_count,
    "recommendation_artifacts": evidence.inventory.recommendation_serving_run_json_count,
}
# render_evidence_report_markdown(evidence)  # full markdown when needed

## Final audit (13.8.9)

Regenerate `resources/benchmark/serving_performance_phase_13_8_final_audit.*` — distinguishes IMPLEMENTED vs MEASURED; no new optimizations.

In [ ]:
from productiq.serving.performance.phase_13_8_audit import build_phase_13_8_final_audit_report

audit = build_phase_13_8_final_audit_report()
{
    "search_measured": next(s for s in audit.sub_phases if s.phase_id == "13.8.3").measurement.value,
    "product_measured": next(s for s in audit.sub_phases if s.phase_id == "13.8.5").measurement.value,
    "integrity_failures": audit.artifact_integrity.run_results_failed + audit.artifact_integrity.sweeps_failed,
}

In [ ]:
from productiq.serving.performance.product_workloads import PRODUCT_SERVING_PERFORMANCE_WORKLOADS

[(w.workload_id, w.path_parameters.get("product_id")) for w in PRODUCT_SERVING_PERFORMANCE_WORKLOADS]

## Product serving benchmark (opt-in)

Requires `resources/processed/product_catalog.parquet` and `PRODUCTIQ_PRODUCT_SERVING_PERFORMANCE_BENCHMARK=1`. Primary suite uses existing-product workloads only; missing-product 404 is tested separately.

```bash
python scripts/run_product_serving_performance_benchmark.py
```

Artifacts: `resources/benchmark/product_serving/`.

In [ ]:
from productiq.serving.catalog_read import InMemoryProductCatalogReadProvider
from productiq.serving.performance.product_benchmark import run_product_serving_performance_suite
from productiq.serving.performance.product_benchmark_config import ProductServingBenchmarkSettings
from productiq.serving.performance.product_workloads import PRODUCT_SERVING_EXISTING_V1
from tests.serving.test_product_service import _sample_record
from productiq.serving.performance.product_workloads import PRODUCT_SERVING_PERFORMANCE_EXISTING_PRODUCT_IDS

catalog = InMemoryProductCatalogReadProvider(
    {pid: _sample_record(pid) for pid in PRODUCT_SERVING_PERFORMANCE_EXISTING_PRODUCT_IDS}
)
suite = run_product_serving_performance_suite(
    catalog,
    settings=ProductServingBenchmarkSettings(warmups=0, iterations=1, concurrency=1),
    workloads=(PRODUCT_SERVING_EXISTING_V1,),
)
suite.service_results[0].latency.mean_ms if suite.service_results[0].latency else None

## Concurrency sweep (13.8.6)

Default levels `[1, 2, 4, 8]` with independent warmups per level. Production product sweep: `PRODUCTIQ_SERVING_CONCURRENCY_BENCHMARK=1`.

In [ ]:
from productiq.api.app import create_app
from productiq.serving.performance.concurrency_benchmark import run_api_concurrency_sweep
from productiq.serving.performance.concurrency_benchmark_config import ConcurrencySweepSettings
from productiq.serving.performance.concurrency_workloads import concurrency_workload_for_endpoint
from productiq.serving.performance.schema import ServingPerformanceEndpoint
from tests.api.fakes import RecordingSearchService

app = create_app(search_service=RecordingSearchService())
sweep = run_api_concurrency_sweep(
    app,
    concurrency_workload_for_endpoint(ServingPerformanceEndpoint.SEARCH),
    ConcurrencySweepSettings(warmups=0, iterations=2, concurrency_levels=(1, 2)),
)
[(r.configuration.concurrency, r.throughput.requests_per_second if r.throughput else None) for r in sweep.run_results]